In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import calendar as cld
import matplotlib.pyplot as plt
import matplotlib.colors
colors_land = plt.cm.terrain(np.linspace(0.25, 1, 256))
import proplot as pplt # New plot library (https://proplot.readthedocs.io/en/latest/)
pplt.rc['savefig.dpi'] = 300 # 1200 is too big! #https://proplot.readthedocs.io/en/latest/basics.html#Creating-figures
from scipy.stats import chi2
from numba import njit,prange
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.05  # previous pdf hatch linewidth
from matplotlib.dates import DateFormatter

In [2]:
ds_T_test = xr.open_dataset('/bettik/castelli/test/ICE.EC-Earth3.EUe.QQp.1979.nc')
ds_T_test

<xarray.Dataset>
Dimensions:  (time: 59, plev: 7, y: 126, x: 201)
Coordinates:
  * plev     (plev) float32 925.0 850.0 800.0 700.0 600.0 500.0 200.0
  * time     (time) datetime64[ns] 1979-01-01T12:00:00 ... 1979-02-28T12:00:00
  * x        (x) float32 -483.0 -476.0 -469.0 -462.0 ... 896.0 903.0 910.0 917.0
  * y        (y) float32 -308.0 -301.0 -294.0 -287.0 ... 546.0 553.0 560.0 567.0
Data variables:
    QQp      (time, plev, y, x) float32 ...
Attributes:
    title:        ICE - Exp: e04 - 19790101
    institution:  ULg (Xavier Fettweis)
    history:      Fri Nov 10 15:58:34 2023: ncks -v QQp ICE.e04.1979.01.01-31...
    netcdf:       4.1.1 of Feb 12 2011 08:44:29 $
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

In [3]:
ds_T_test2 = xr.open_dataset('/bettik/castelli/test/ICE.EC-Earth3.EUe.QQp.1979_test.nc')
ds_T_test2

<xarray.Dataset>
Dimensions:  (time: 59, plev: 7, y: 126, x: 201)
Coordinates:
  * plev     (plev) float32 925.0 850.0 800.0 700.0 600.0 500.0 200.0
  * time     (time) datetime64[ns] 1979-01-01T12:00:00 ... 1979-02-28T12:00:00
  * x        (x) float32 -483.0 -476.0 -469.0 -462.0 ... 896.0 903.0 910.0 917.0
  * y        (y) float32 -308.0 -301.0 -294.0 -287.0 ... 546.0 553.0 560.0 567.0
Data variables:
    QQp      (time, plev, y, x) float32 ...
Attributes:
    title:        ICE - Exp: e04 - 19790101
    institution:  ULg (Xavier Fettweis)
    history:      Fri Nov 10 15:58:34 2023: ncks -v QQp ICE.e04.1979.01.01-31...
    netcdf:       4.1.1 of Feb 12 2011 08:44:29 $
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

In [10]:
(ds_T_test == ds_T_test2)

<xarray.Dataset>
Dimensions:  (plev: 7, time: 59, x: 201, y: 126)
Coordinates:
  * plev     (plev) float32 925.0 850.0 800.0 700.0 600.0 500.0 200.0
  * time     (time) datetime64[ns] 1979-01-01T12:00:00 ... 1979-02-28T12:00:00
  * x        (x) float32 -483.0 -476.0 -469.0 -462.0 ... 896.0 903.0 910.0 917.0
  * y        (y) float32 -308.0 -301.0 -294.0 -287.0 ... 546.0 553.0 560.0 567.0
Data variables:
    QQp      (time, plev, y, x) bool True True True True ... True True True True

In [6]:
ds_T_test3 = xr.open_dataset('/bettik/castelli/test/ICE.EC-Earth3.EUe.TTp.01.nc')
ds_T_test3

<xarray.Dataset>
Dimensions:  (time: 31, plev: 7, y: 126, x: 201)
Coordinates:
  * plev     (plev) float32 925.0 850.0 800.0 700.0 600.0 500.0 200.0
  * time     (time) datetime64[ns] 1979-01-01T12:00:00 ... 1979-01-31T12:00:00
  * x        (x) float32 -483.0 -476.0 -469.0 -462.0 ... 896.0 903.0 910.0 917.0
  * y        (y) float32 -308.0 -301.0 -294.0 -287.0 ... 546.0 553.0 560.0 567.0
Data variables:
    TTp      (time, plev, y, x) float32 ...
Attributes:
    title:        ICE - Exp: e04 - 19790101
    institution:  ULg (Xavier Fettweis)
    history:      Thu Nov  9 16:48:42 2023: ncks -v TTp ICE.e04.1979.01.01-31...
    netcdf:       4.1.1 of Feb 12 2011 08:44:29 $
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

In [9]:
(ds_T_test == ds_T_test3)


<xarray.Dataset>
Dimensions:  (time: 31, plev: 7, x: 201, y: 126)
Coordinates:
  * time     (time) datetime64[ns] 1979-01-01T12:00:00 ... 1979-01-31T12:00:00
  * plev     (plev) float32 925.0 850.0 800.0 700.0 600.0 500.0 200.0
  * x        (x) float32 -483.0 -476.0 -469.0 -462.0 ... 896.0 903.0 910.0 917.0
  * y        (y) float32 -308.0 -301.0 -294.0 -287.0 ... 546.0 553.0 560.0 567.0
Data variables:
    *empty*

In [2]:
ds_QQp = xr.open_dataset('/bettik/castelli/data/MAR-EC-Earth3/HIST/ICE.EC-Earth3_HIST.EUe.QQp.2000.nc')
ds_QQp

<xarray.Dataset>
Dimensions:  (time: 366, plev: 7, y: 126, x: 201)
Coordinates:
  * plev     (plev) float32 925.0 850.0 800.0 700.0 600.0 500.0 200.0
  * time     (time) datetime64[ns] 2000-01-01T12:00:00 ... 2000-12-31T12:00:00
  * x        (x) float32 -483.0 -476.0 -469.0 -462.0 ... 896.0 903.0 910.0 917.0
  * y        (y) float32 -308.0 -301.0 -294.0 -287.0 ... 546.0 553.0 560.0 567.0
Data variables:
    QQp      (time, plev, y, x) float32 ...
Attributes:
    title:        ICE - Exp: e08 - 20000101
    institution:  ULg (Xavier Fettweis)
    history:      Tue Nov 14 17:14:38 2023: ncks -v QQp /bettik/menegozm/MARo...
    netcdf:       4.1.1 of Feb 12 2011 08:44:29 $
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...